# Plot ProDy ANM frequency results in Colab

This notebook is for **plotting only**. It does **not** run ProDy and does **not** repeat the HPC calculation.

Use it after you copy the HPC output folder to your local computer. The main required input file is:

```text
frequencies_long.csv
```

Optional input file:

```text
summary.csv
```

You can upload either individual CSV files or a ZIP file containing your results folder, for example `results_motif_2.zip`.


In [ ]:
# Install plotting dependencies. ProDy is not needed for this notebook.
%pip -q install pandas numpy matplotlib

In [ ]:
from pathlib import Path
import os
import re
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

try:
    from google.colab import files, drive
    IN_COLAB = True
except Exception:
    files = None
    drive = None
    IN_COLAB = False

# Plot settings. You can edit these and rerun the plotting cells.
BINS = 120        # Number of histogram bins
ZOOM_MAX = 3.0    # THz upper limit for zoomed plots
DPI = 300         # Image resolution
OUTPUT_DIR = Path("plots_motif47")

print("Running in Colab:", IN_COLAB)
print("Output folder:", OUTPUT_DIR)

## Upload your HPC results

Run the next cell and upload one of these:

- `frequencies_long.csv` by itself, or
- both `frequencies_long.csv` and `summary.csv`, or
- a ZIP file containing the result folder from the HPC.

The compute script writes `frequencies_long.csv`, `summary.csv`, `frequencies.npz`, and `run_manifest.json`; this notebook mainly uses `frequencies_long.csv`.


In [ ]:
UPLOAD_DIR = Path("/content/prody_results") if IN_COLAB else Path("prody_results")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    print("Upload frequencies_long.csv, summary.csv, or a .zip file containing them.")
    uploaded = files.upload()
    for filename, data in uploaded.items():
        destination = UPLOAD_DIR / filename
        destination.write_bytes(data)
        print(f"Uploaded: {destination}")
        if destination.suffix.lower() == ".zip":
            extract_dir = UPLOAD_DIR / destination.stem
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(destination, "r") as zf:
                zf.extractall(extract_dir)
            print(f"Extracted ZIP to: {extract_dir}")
else:
    print("Not running in Colab. Put your result files in:", UPLOAD_DIR.resolve())

# Find the input files automatically.
search_roots = [UPLOAD_DIR, Path.cwd()]
frequency_candidates = []
summary_candidates = []
for root in search_roots:
    if root.exists():
        frequency_candidates.extend(root.rglob("frequencies_long.csv"))
        summary_candidates.extend(root.rglob("summary.csv"))

# Remove duplicates while preserving order.
def unique_paths(paths):
    seen = set()
    out = []
    for path in paths:
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            out.append(path)
    return out

frequency_candidates = unique_paths(frequency_candidates)
summary_candidates = unique_paths(summary_candidates)

if not frequency_candidates:
    raise FileNotFoundError(
        "Could not find frequencies_long.csv. Upload it directly or upload a ZIP containing it."
    )

FREQUENCIES_CSV = frequency_candidates[0]
SUMMARY_CSV = summary_candidates[0] if summary_candidates else None

print("Using frequency file:", FREQUENCIES_CSV)
print("Using summary file:", SUMMARY_CSV if SUMMARY_CSV else "not found; this is OK")

### Optional: use Google Drive instead of upload

Skip this if you used the upload cell above. To use Drive, run the cell below, then edit `FREQUENCIES_CSV` to the path of your file in Google Drive.


In [ ]:
# Optional Google Drive workflow.
# Uncomment and edit these lines if your result files are in Google Drive.

# drive.mount('/content/drive')
# FREQUENCIES_CSV = Path('/content/drive/MyDrive/results_motif_2/frequencies_long.csv')
# SUMMARY_CSV = Path('/content/drive/MyDrive/results_motif_2/summary.csv')
# print(FREQUENCIES_CSV.exists(), SUMMARY_CSV.exists())

## Load and inspect the data

In [ ]:
REQUIRED_COLUMNS = {"label", "mode_index", "eigenvalue", "frequency_thz"}


def read_frequency_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    df = pd.read_csv(path)
    missing = REQUIRED_COLUMNS.difference(df.columns)
    if missing:
        raise ValueError(
            f"Input CSV is missing required columns: {sorted(missing)}. "
            f"Found columns: {list(df.columns)}"
        )

    df = df.copy()
    df["label"] = df["label"].astype(str)
    df["mode_index"] = pd.to_numeric(df["mode_index"], errors="coerce")
    df["eigenvalue"] = pd.to_numeric(df["eigenvalue"], errors="coerce")
    df["frequency_thz"] = pd.to_numeric(df["frequency_thz"], errors="coerce")
    df = df.dropna(subset=["label", "mode_index", "frequency_thz"])
    df = df[df["frequency_thz"] >= 0]

    if df.empty:
        raise ValueError("No usable frequency rows were found after cleaning the data.")

    return df


df = read_frequency_table(FREQUENCIES_CSV)

print(f"Rows: {len(df):,}")
print(f"Proteins/structures: {df['label'].nunique()}")
print(f"Frequency range: {df['frequency_thz'].min():.6g} to {df['frequency_thz'].max():.6g} THz")
print("Labels:", ", ".join(sorted(df["label"].unique())))

display(df.head())

if SUMMARY_CSV and Path(SUMMARY_CSV).exists():
    summary_df = pd.read_csv(SUMMARY_CSV)
    print("Summary file loaded:", SUMMARY_CSV)
    display(summary_df)
else:
    summary_df = None
    print("No summary.csv loaded; statistics will be computed from frequencies_long.csv.")

## Plotting functions

In [ ]:
def safe_name(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_") or "protein"


def save_figure(path: Path, dpi: int = DPI, show: bool = True) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"Wrote: {path}")
    if show:
        display(Image(filename=str(path)))


def plot_vdos_overlay(
    df: pd.DataFrame,
    output_path: Path,
    bins: int = BINS,
    dpi: int = DPI,
    f_min: float | None = None,
    f_max: float | None = None,
    title: str = "VDOS overlay",
    show: bool = True,
) -> None:
    work = df.copy()
    if f_min is not None:
        work = work[work["frequency_thz"] >= f_min]
    if f_max is not None:
        work = work[work["frequency_thz"] <= f_max]

    if work.empty:
        print(f"Skipping {output_path}: no data in selected range")
        return

    lo = float(work["frequency_thz"].min()) if f_min is None else float(f_min)
    hi = float(work["frequency_thz"].max()) if f_max is None else float(f_max)
    if hi <= lo:
        hi = lo + 1e-9

    edges = np.linspace(lo, hi, bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    plt.figure(figsize=(9, 5.5))
    for label, group in work.groupby("label", sort=True):
        values = group["frequency_thz"].to_numpy(dtype=float)
        if len(values) == 0:
            continue
        density, _ = np.histogram(values, bins=edges, density=True)
        plt.plot(centers, density, linewidth=1.8, label=label)

    plt.xlabel("Frequency (THz)")
    plt.ylabel("Normalized density")
    plt.title(title)
    plt.legend(fontsize=8, ncol=2)
    plt.grid(True, alpha=0.3)
    save_figure(output_path, dpi=dpi, show=show)


def plot_frequency_vs_mode(df: pd.DataFrame, output_path: Path, dpi: int = DPI, show: bool = True) -> None:
    plt.figure(figsize=(9, 5.5))
    for label, group in df.groupby("label", sort=True):
        group = group.sort_values("mode_index")
        plt.plot(group["mode_index"], group["frequency_thz"], linewidth=1.2, label=label)

    plt.xlabel("Mode index")
    plt.ylabel("Frequency (THz)")
    plt.title("Frequency vs mode index")
    plt.legend(fontsize=8, ncol=2)
    plt.grid(True, alpha=0.3)
    save_figure(output_path, dpi=dpi, show=show)


def plot_boxplot(df: pd.DataFrame, output_path: Path, dpi: int = DPI, zoom_max: float | None = None, show: bool = True) -> None:
    work = df.copy()
    title = "Frequency distribution by protein"
    if zoom_max is not None:
        work = work[work["frequency_thz"] <= zoom_max]
        title = f"Frequency distribution by protein, 0-{zoom_max:g} THz"

    if work.empty:
        print(f"Skipping {output_path}: no data in selected range")
        return

    labels = sorted(work["label"].unique())
    values = [work.loc[work["label"] == label, "frequency_thz"].to_numpy(dtype=float) for label in labels]

    plt.figure(figsize=(max(8, 0.7 * len(labels)), 5.5))
    plt.boxplot(values, tick_labels=labels, showfliers=False)
    plt.xlabel("Protein")
    plt.ylabel("Frequency (THz)")
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.grid(True, axis="y", alpha=0.3)
    save_figure(output_path, dpi=dpi, show=show)


def build_statistics(df: pd.DataFrame) -> pd.DataFrame:
    stats = (
        df.groupby("label")["frequency_thz"]
        .agg(
            n_modes="count",
            min_frequency_thz="min",
            max_frequency_thz="max",
            mean_frequency_thz="mean",
            median_frequency_thz="median",
            std_frequency_thz="std",
        )
        .reset_index()
    )

    low_counts = df.groupby("label")["frequency_thz"].apply(lambda s: int((s < 0.1).sum()))
    thz_counts = df.groupby("label")["frequency_thz"].apply(lambda s: int(((s >= 0.1) & (s <= 3.0)).sum()))
    high_counts = df.groupby("label")["frequency_thz"].apply(lambda s: int((s > 3.0).sum()))

    stats["modes_microwave_lt_0_1_thz"] = stats["label"].map(low_counts).astype(int)
    stats["modes_thz_0_1_to_3_0_thz"] = stats["label"].map(thz_counts).astype(int)
    stats["modes_beyond_3_0_thz"] = stats["label"].map(high_counts).astype(int)
    return stats


def plot_summary_mean(stats: pd.DataFrame, output_path: Path, dpi: int = DPI, show: bool = True) -> None:
    stats = stats.sort_values("mean_frequency_thz")
    plt.figure(figsize=(max(8, 0.7 * len(stats)), 5.5))
    plt.bar(stats["label"], stats["mean_frequency_thz"])
    plt.xlabel("Protein")
    plt.ylabel("Mean frequency (THz)")
    plt.title("Mean ANM frequency by protein")
    plt.xticks(rotation=45, ha="right")
    plt.grid(True, axis="y", alpha=0.3)
    save_figure(output_path, dpi=dpi, show=show)


def plot_mode_count_ranges(df: pd.DataFrame, output_path: Path, dpi: int = DPI, show: bool = True) -> pd.DataFrame:
    labels = sorted(df["label"].unique())
    counts = []
    for label in labels:
        values = df.loc[df["label"] == label, "frequency_thz"].to_numpy(dtype=float)
        counts.append(
            {
                "label": label,
                "<0.1 THz": int(np.sum(values < 0.1)),
                "0.1-3 THz": int(np.sum((values >= 0.1) & (values <= 3.0))),
                ">3 THz": int(np.sum(values > 3.0)),
            }
        )
    count_df = pd.DataFrame(counts)
    x = np.arange(len(labels))
    width = 0.25

    plt.figure(figsize=(max(9, 0.8 * len(labels)), 5.5))
    plt.bar(x - width, count_df["<0.1 THz"], width, label="<0.1 THz")
    plt.bar(x, count_df["0.1-3 THz"], width, label="0.1-3 THz")
    plt.bar(x + width, count_df[">3 THz"], width, label=">3 THz")
    plt.xticks(x, labels, rotation=45, ha="right")
    plt.xlabel("Protein")
    plt.ylabel("Number of modes")
    plt.title("Mode counts by frequency range")
    plt.legend()
    plt.grid(True, axis="y", alpha=0.3)
    save_figure(output_path, dpi=dpi, show=show)
    return count_df


def plot_individual_histograms(
    df: pd.DataFrame,
    output_dir: Path,
    bins: int = BINS,
    dpi: int = DPI,
    zoom_max: float | None = ZOOM_MAX,
    show: bool = False,
) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for label, group in df.groupby("label", sort=True):
        values = group["frequency_thz"].to_numpy(dtype=float)
        label_safe = safe_name(label)

        plt.figure(figsize=(7.5, 5))
        plt.hist(values, bins=bins, density=True)
        plt.xlabel("Frequency (THz)")
        plt.ylabel("Normalized density")
        plt.title(f"Frequency histogram: {label}")
        plt.grid(True, alpha=0.3)
        save_figure(output_dir / f"{label_safe}_hist_full.png", dpi=dpi, show=show)

        if zoom_max is not None:
            zoom_values = values[values <= zoom_max]
            if len(zoom_values) > 0:
                plt.figure(figsize=(7.5, 5))
                plt.hist(zoom_values, bins=bins, density=True)
                plt.xlabel("Frequency (THz)")
                plt.ylabel("Normalized density")
                plt.title(f"Frequency histogram: {label}, 0-{zoom_max:g} THz")
                plt.grid(True, alpha=0.3)
                save_figure(output_dir / f"{label_safe}_hist_0_to_{zoom_max:g}_THz.png", dpi=dpi, show=show)

## Generate all plots

The main plots are displayed in the notebook and saved as `.png` files in `plots_motif47/`. Individual histograms are saved in `plots_motif47/individual_histograms/` but not displayed inline by default.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stats = build_statistics(df)
stats_path = OUTPUT_DIR / "statistics_from_frequencies.csv"
stats.to_csv(stats_path, index=False)
print("Wrote:", stats_path)
display(stats)

plot_vdos_overlay(
    df,
    OUTPUT_DIR / "vdos_overlay_full_range.png",
    bins=BINS,
    dpi=DPI,
    title="VDOS overlay, full frequency range",
)

plot_vdos_overlay(
    df,
    OUTPUT_DIR / f"vdos_overlay_0_to_{ZOOM_MAX:g}_THz.png",
    bins=BINS,
    dpi=DPI,
    f_min=0.0,
    f_max=ZOOM_MAX,
    title=f"VDOS overlay, 0-{ZOOM_MAX:g} THz",
)

plot_frequency_vs_mode(df, OUTPUT_DIR / "frequency_vs_mode_index.png", dpi=DPI)
plot_boxplot(df, OUTPUT_DIR / "boxplot_full_range.png", dpi=DPI)
plot_boxplot(df, OUTPUT_DIR / f"boxplot_0_to_{ZOOM_MAX:g}_THz.png", dpi=DPI, zoom_max=ZOOM_MAX)
plot_summary_mean(stats, OUTPUT_DIR / "mean_frequency_by_protein.png", dpi=DPI)

count_df = plot_mode_count_ranges(df, OUTPUT_DIR / "mode_counts_by_frequency_range.png", dpi=DPI)
display(count_df)

plot_individual_histograms(
    df,
    OUTPUT_DIR / "individual_histograms",
    bins=BINS,
    dpi=DPI,
    zoom_max=ZOOM_MAX,
    show=False,
)

print("Done. Plot files are in:", OUTPUT_DIR.resolve())

## Download the plots

Run this cell to create a ZIP file of the plots and download it from Colab.


In [ ]:
zip_base = OUTPUT_DIR.as_posix()
zip_path = shutil.make_archive(zip_base, "zip", root_dir=OUTPUT_DIR)
print("Created:", zip_path)

if IN_COLAB:
    files.download(zip_path)
else:
    print("Download manually from:", zip_path)